# River chance operators and suit symmetry

This notebook is a hands-on tour of the exact river kernel. It keeps every calculation in the fixed 1,326-dimensional private-hand basis and explores the action of the 24 suit permutations.

Run the cells from top to bottom. Change the board and example ranges to explore other river situations.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Make the notebook work whether Jupyter starts in the repository root or experiments/.
project_root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "cards.py").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from cards import hand_names, make_hand
from hand_space import HAND_INDEX, PRIVATE_HANDS, compatible_hand_indices, hand_label
from river_kernel import result_matrix, river_kernel
from symmetry import (
    HAND_PERMUTATIONS,
    SUIT_PERMUTATIONS,
    board_orbit_count,
    canonical_board,
    hand_permutation,
    permute_cards,
    private_hand_orbits,
    stabilizer,
)

def cards(text):
    return make_hand(text.split())

def names(card_set):
    return " ".join(hand_names(card_set))

## The ambient private-hand space

Every calculation uses the same coordinates, including hands blocked by the board. This is what allows operators from different boards to be compared directly.

In [ ]:
print(f"Ambient dimension: {len(PRIVATE_HANDS):,}")
print("First ten basis hands:", [hand_label(hand) for hand in PRIVATE_HANDS[:10]])

example_hand = cards("As Kd")
print(f"Coordinate of As Kd: {HAND_INDEX[example_hand]}")

## Build the river kernel

`C[i, j]` records whether two private hands can coexist with the board. `D[i, j]` is `-1`, `0`, or `+1` when the row hand loses, ties, or wins. Incompatible entries are zero. The doubled-equity result is `R = C + D`.

In [ ]:
board = cards("2c 7d 9h Js 3c")  # Edit this five-card board.
C, D = river_kernel(board)
R = C.astype(np.int8) + D

print("Board:", names(board))
print(f"Compatible private hands: {len(compatible_hand_indices(board)):,}")
print(f"Compatible ordered matchups: {int(C.sum()):,}")
print("C symmetric:     ", np.array_equal(C, C.T))
print("D skew-symmetric:", np.array_equal(D, -D.T))
print("D values:        ", np.unique(D))

## Inspect individual matchups

The pair `(C, D)` distinguishes incompatibility, loss, tie, and win without putting strings into the matrix.

In [ ]:
def inspect_matchup(first_text, second_text):
    first = HAND_INDEX[cards(first_text)]
    second = HAND_INDEX[cards(second_text)]
    state = {
        (False, 0): "incompatible",
        (True, -1): "loss",
        (True, 0): "tie",
        (True, 1): "win",
    }[(bool(C[first, second]), int(D[first, second]))]
    return {"first": first_text, "second": second_text, "C": bool(C[first, second]),
            "D": int(D[first, second]), "R": int(R[first, second]), "state": state}

inspect_matchup("Tc Ad", "9c Kd")

## Range-versus-range equity

Ranges are non-negative mass vectors in the ambient space. Their normalized equity is

$$\operatorname{Equity}_B(x,y)=\frac12+\frac{x^T D_B y}{2x^T C_B y}.$$

In [ ]:
def range_vector(weighted_hands):
    vector = np.zeros(len(PRIVATE_HANDS), dtype=float)
    for hand_text, weight in weighted_hands.items():
        vector[HAND_INDEX[cards(hand_text)]] = weight
    return vector

def range_equity(x, y, compatibility=C, dominance=D):
    compatible_mass = float(x @ compatibility @ y)
    if compatible_mass == 0:
        raise ValueError("The ranges have no compatible mass on this board")
    edge = float(x @ dominance @ y)
    return 0.5 + edge / (2 * compatible_mass)

hero = range_vector({"Tc Ad": 1, "9c Kd": 1, "Ac Ah": 2})
villain = range_vector({"8c 8d": 1, "Qc Qd": 2, "4c 5d": 1})

print(f"Hero equity:    {range_equity(hero, villain):.3%}")
print(f"Villain equity: {range_equity(villain, hero):.3%}")

## Visualize a compatible submatrix

This shows a small slice using only hands compatible with the board. Blue cells favour the row hand, red cells favour the column hand, and white cells tie or cannot coexist.

In [ ]:
sample = np.asarray(compatible_hand_indices(board)[:150])
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(D[np.ix_(sample, sample)], cmap="coolwarm_r", vmin=-1, vmax=1)
ax.set(title="Dominance operator (first 150 board-compatible hands)", xlabel="Column hand", ylabel="Row hand")
fig.colorbar(image, ax=ax, ticks=[-1, 0, 1], label="loss / tie / win")
plt.show()

## Global suit symmetry

The full group is $S_4$, acting by permuting clubs, diamonds, hearts, and spades. Its action derives both familiar counting results rather than hard-coding them.

In [ ]:
orbits = private_hand_orbits()
print(f"Suit permutations:          {len(SUIT_PERMUTATIONS)}")
print(f"Private-hand suit orbits:  {len(orbits)}")
print(f"Five-card board orbits:    {board_orbit_count(5):,}")
print("First five hand orbits:")
for orbit in orbits[:5]:
    print([hand_label(PRIVATE_HANDS[index]) for index in orbit])

## Canonical boards and stabilizers

A board's canonical representative is the smallest bitset in its suit orbit. A fixed board only commutes with its stabilizer, not generally with all of $S_4$.

In [ ]:
canonical = canonical_board(board)
print("Current board:   ", names(board))
print("Canonical board: ", names(canonical))
print("Stabilizer size: ", len(stabilizer(board)))

monotone_board = cards("2c 5c 8c Jc Ac")
print("Monotone board stabilizer size:", len(stabilizer(monotone_board)))

## Verify equivariance

For a suit permutation $g$, the transported operator satisfies $D_{gB}=\rho(g)D_B\rho(g)^{-1}$, and likewise for $C$. The index array stores the map $i\mapsto g(i)$.

In [ ]:
g = (2, 0, 3, 1)  # old suit index -> new suit index
moved_board = permute_cards(board, g)
permuted_indices = hand_permutation(g)
C_moved, D_moved = river_kernel(moved_board)
coordinates = np.ix_(permuted_indices, permuted_indices)

print("Original board: ", names(board))
print("Moved board:    ", names(moved_board))
print("C equivariant:  ", np.array_equal(C_moved[coordinates], C))
print("D equivariant:  ", np.array_equal(D_moved[coordinates], D))

## Project a vector onto the invariant subspace

Averaging any range vector over all 24 suit permutations projects it onto the 169-dimensional trivial component. This is the first representation-theoretic operation, implemented without dense permutation matrices.

In [ ]:
rng = np.random.default_rng(7)
vector = rng.random(len(PRIVATE_HANDS))
invariant = np.zeros_like(vector)
for permutation_indices in HAND_PERMUTATIONS:
    moved = np.empty_like(vector)
    moved[permutation_indices] = vector
    invariant += moved
invariant /= len(HAND_PERMUTATIONS)

max_errors = []
for permutation_indices in HAND_PERMUTATIONS:
    moved = np.empty_like(invariant)
    moved[permutation_indices] = invariant
    max_errors.append(np.max(np.abs(moved - invariant)))

print(f"Largest invariance error: {max(max_errors):.3e}")
print("The projected vector is constant on every hand orbit:", all(
    np.allclose(invariant[list(orbit)], invariant[orbit[0]]) for orbit in orbits
))

## Ideas to try next

- Replace `board` with paired, monotone, and rainbow boards and compare their stabilizers.
- Build weighted ranges from solver outputs and compare the compatibility mass $x^T C y$ with the directional edge $x^T D y$.
- Group compatible hands into stabilizer orbits for a monotone board.
- Compare singular values of `D` across canonical board textures.
- Average empirical ranges over the suit action and measure the residual outside the invariant subspace.